# 07 - Ensemble & Scoreboard: the Inertia strategy

**Inertia, Momentum Timing** - Sprint 4 (final)

Sprints 1-3 established three regime-timing approaches:

- **Approach A** (Ridge): linear expected-return model with expanded features.
- **Approach B** (GBM shield): classify crashes, gate exposure on/off.
- **Approach C** (HMM): unsupervised 2-state regime detection.

None of them beat Daniel-Moskowitz OOS on Sharpe cleanly. But they produce **methodologically orthogonal** signals: regression vs classification vs unsupervised. That orthogonality is ensemble fuel.

**Inertia's production strategy** (proposed):

$$
w^{\text{Inertia}}_t \;=\; 0.5 \cdot w^{DM}_t \;+\; 0.25 \cdot \mathbb{1}\{p^{B}_{\text{crash},t} < 0.05\} \;+\; 0.25 \cdot P^{C}(\text{normal}_t)
$$

Weights are then clipped to [-1, +3] and held for one month at a time. 20 bps one-way cost assumption.

**Target:** a Sharpe statistically distinguishable from DM OOS, with lower vol and drawdown.

Window: **OOS from 1960-01**, 66-year backtest, N = 793 months.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from hmmlearn.hmm import GaussianHMM

from src.features import build_features, DM_FEATURES, MARKET_FEATURES
from src.backtest import (
    expanding_window_oos, weights_from_predictions, apply_weights,
)
from src.evaluation import (
    perf_table, sharpe_bootstrap_ci, sharpe_diff_ci, alpha_table, subsample_table,
)
from src.data import get_factor_panel
from src.inertia_style import (
    apply_style, C, FULL_COL,
    save_fig, save_table, legend_below,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1960-01-01'

HMM_OBS = ['MKT_RF', 'mkt_vol6', 'mom_var6']

## 1. Build feature panels and the three signals

In [2]:
df = build_features(include_macro=False)
feat_expanded = DM_FEATURES + MARKET_FEATURES
complete = df[feat_expanded + ['UMD_next','UMD']].dropna()
hmm_complete = df[HMM_OBS + ['UMD_next','UMD']].dropna()

print(f'Feature panel: {df.shape}, {df.index.min().date()} -> {df.index.max().date()}')

Feature panel: (1179, 14), 1927-12-31 -> 2026-02-28


In [3]:
# === DM benchmark ===
class OLSModel:
    def __init__(self, res): self.res = res
    def predict(self, X):
        return self.res.predict(sm.add_constant(X, has_constant='add'))
def fit_ols(X, y): return OLSModel(sm.OLS(y, sm.add_constant(X)).fit())

dm_preds = expanding_window_oos(df, DM_FEATURES, 'UMD_next', fit_fn=fit_ols,
                                 oos_start=OOS_START, refit_months=12, min_train_months=120)
dm_w, _ = weights_from_predictions(dm_preds, df['UMD'], cap=(-1.0, 3.0))

# === Approach A: Ridge ===
def fit_ridge(X, y):
    return Pipeline([('scale', StandardScaler()),
                     ('ridge', RidgeCV(alphas=np.logspace(-2, 2, 13), cv=5))]).fit(X, y)
A_preds = expanding_window_oos(complete, feat_expanded, 'UMD_next', fit_fn=fit_ridge,
                                oos_start=OOS_START, refit_months=12, min_train_months=120)
A_w, _ = weights_from_predictions(A_preds, df['UMD'], cap=(-1.0, 3.0))

# === Approach B: GBM shield ===
class ClfWrap:
    def __init__(self, clf): self.clf = clf
    def predict(self, X): return self.clf.predict_proba(X)[:, 1]
def fit_gb(X, y_ret):
    thr = float(np.quantile(y_ret, 0.10))
    y_crash = (y_ret < thr).astype(int)
    return ClfWrap(GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, random_state=42,
    ).fit(X, y_crash))

B_p_crash = expanding_window_oos(complete, feat_expanded, 'UMD_next', fit_fn=fit_gb,
                                   oos_start=OOS_START, refit_months=12, min_train_months=120)
shield = pd.Series(np.where(B_p_crash < 0.05, 1.0, 0.0), index=B_p_crash.index, name='shield')

# === Approach C: HMM ===
class HMMFilter:
    def __init__(self, hmm, ns, trX):
        self.hmm = hmm; self.ns = ns
        self._last = hmm.predict_proba(trX)[-1]
    def predict(self, X):
        X = np.asarray(X); probs = np.zeros(len(X))
        p = self._last.copy(); T = self.hmm.transmat_
        for i in range(len(X)):
            p = p @ T
            le = self.hmm._compute_log_likelihood(X[i:i+1])[0]
            e = np.exp(le - le.max()); p = p * e; p /= max(p.sum(), 1e-20)
            probs[i] = p[self.ns]
        return probs
def fit_hmm(X, y):
    hmm = GaussianHMM(n_components=2, covariance_type='full', n_iter=100, random_state=42).fit(X)
    st = hmm.predict(X)
    rets = [np.mean(y[st == s]) if (st == s).any() else -np.inf for s in range(2)]
    return HMMFilter(hmm, int(np.argmax(rets)), X)

p_normal = expanding_window_oos(hmm_complete, HMM_OBS, 'UMD_next', fit_fn=fit_hmm,
                                  oos_start=OOS_START, refit_months=12, min_train_months=120)
print(f'All three signals built. Aligning...')
idx = dm_w.index.intersection(A_w.index).intersection(shield.index).intersection(p_normal.index)
print(f'Shared OOS: {len(idx)} months, {idx.min().date()} -> {idx.max().date()}')

Model is not converging.  Current: 2312.7499974784355 is not greater than 2313.2229916404613. Delta is -0.4729941620257705


Model is not converging.  Current: 2465.274760198923 is not greater than 2465.2756372355093. Delta is -0.000877036586189206


Model is not converging.  Current: 2550.8553711831664 is not greater than 2550.983468705091. Delta is -0.12809752192470114


Model is not converging.  Current: 2808.7604208895873 is not greater than 2808.9736942598443. Delta is -0.2132733702569567


Model is not converging.  Current: 2890.5226020769073 is not greater than 2890.533522501803. Delta is -0.010920424895630276


...

Model is not converging.  Current: 6608.331676043131 is not greater than 6608.828753987554. Delta is -0.4970779444229265


Model is not converging.  Current: 6723.307470832543 is not greater than 6724.013158601045. Delta is -0.7056877685017753


Model is not converging.  Current: 6814.955569752395 is not greater than 6815.496293300247. Delta is -0.5407235478523944


Model is not converging.  Current: 6906.580509748091 is not greater than 6906.729563999849. Delta is -0.1490542517585709


All three signals built. Aligning...
Shared OOS: 793 months, 1960-01-31 -> 2026-01-31


## 2. Construct the Inertia ensemble

In [4]:
dm_w_a     = dm_w.loc[idx]
A_w_a      = A_w.loc[idx]
shield_a   = shield.loc[idx]
p_normal_a = p_normal.loc[idx]

# Final Inertia strategy: 0.5 DM + 0.25 Shield + 0.25 HMM, clipped to [-1, +3]
inertia_w = (0.5 * dm_w_a
             + 0.25 * shield_a
             + 0.25 * p_normal_a.clip(0, 1)).clip(-1, 3).rename('inertia_weight')

print(f'Inertia weight: mean={inertia_w.mean():.2f}, range=[{inertia_w.min():.2f}, {inertia_w.max():.2f}]')
print(f'  % short:  {(inertia_w < 0).mean()*100:.1f}%')
print(f'  % above 1 (levered long): {(inertia_w > 1).mean()*100:.1f}%')

Inertia weight: mean=0.92, range=[-0.06, 1.98]
  % short:  4.5%
  % above 1 (levered long): 44.6%


## 3. Run the backtest

In [5]:
COST = 20.0

def backtest(w, name):
    back = apply_weights(w, df['UMD'], cost_bps_oneway=COST)
    back.name = name
    return back

static_r  = df['UMD_next'].reindex(idx)
dm_back   = backtest(dm_w_a,  'DM')
A_back    = backtest(A_w_a,   'A_Ridge')
B_back    = backtest(shield_a,'B_Shield')
C_back    = backtest(p_normal_a.clip(0, 1), 'C_HMM')
IN_back   = backtest(inertia_w, 'Inertia')

returns = {
    'Static UMD':  static_r,
    'DM OOS':      dm_back['r_net'],
    'A (Ridge)':   A_back['r_net'],
    'B (Shield)':  B_back['r_net'],
    'C (HMM)':     C_back['r_net'],
    'Inertia':     IN_back['r_net'],
}

## 4. Headline scoreboard

In [6]:
perf = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(perf, '07_scoreboard_performance', out_dir=TABLE_DIR)
perf

  (skipped tex for 07_scoreboard_performance: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/07_scoreboard_performance.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static UMD,793,0.0747,0.1422,0.5255,-1.3193,10.0409,-0.5782
DM OOS,793,0.1048,0.1419,0.7383,-0.3393,4.4136,-0.3394
A (Ridge),793,0.0972,0.1404,0.6925,-0.1474,7.1891,-0.3167
B (Shield),793,0.0514,0.0819,0.6268,0.4313,7.1331,-0.2352
C (HMM),793,0.0774,0.1071,0.7225,-0.4021,5.7751,-0.3063
Inertia,793,0.0852,0.1070,0.7962,-0.3702,2.9528,-0.2200


In [7]:
# Sharpe CIs
boot = {name: sharpe_bootstrap_ci(r, n_boot=2000) for name, r in returns.items()}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '07_scoreboard_sharpe_ci', out_dir=TABLE_DIR)
boot_df

  (skipped tex for 07_scoreboard_sharpe_ci: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/07_scoreboard_sharpe_ci.{csv,md}


,sharpe,ci_low,ci_high
Static UMD,0.5255,0.2466,0.7944
DM OOS,0.7383,0.4667,0.9701
A (Ridge),0.6925,0.4127,0.9208
B (Shield),0.6268,0.3899,0.8560
C (HMM),0.7225,0.4611,0.9559
Inertia,0.7962,0.5300,1.0233


## 5. Paired Sharpe differences vs DM

The real test: does each strategy's Sharpe statistically **exceed** DM's? Paired block-bootstrap CI for SR(X) - SR(DM).

In [8]:
diffs = {}
for name, r in returns.items():
    if name == 'DM OOS':
        continue
    d = sharpe_diff_ci(r, returns['DM OOS'], n_boot=5000)
    diffs[name] = d
diff_df = pd.DataFrame(diffs).T[['diff','ci_low','ci_high','p_value']]
diff_df['beats_DM_95pct'] = diff_df['ci_low'] > 0
save_table(diff_df, '07_paired_sharpe_diff_vs_dm', out_dir=TABLE_DIR)
diff_df

  (skipped tex for 07_paired_sharpe_diff_vs_dm: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/07_paired_sharpe_diff_vs_dm.{csv,md}


,diff,ci_low,ci_high,p_value,beats_DM_95pct
Static UMD,-0.2128,-0.4589,0.0590,0.1404,False
A (Ridge),-0.0458,-0.1512,0.0593,0.4136,False
B (Shield),-0.1115,-0.2851,0.0994,0.3396,False
C (HMM),-0.0158,-0.2038,0.1819,0.9404,False
Inertia,0.0579,0.0029,0.1191,0.0408,True


## 6. Alphas vs FF5+UMD

In [9]:
factor_panel = get_factor_panel()
alpha_df = alpha_table(
    {k: v for k, v in returns.items() if k != 'Static UMD'},
    factor_panel, spec='FF5_UMD',
)
save_table(alpha_df, '07_scoreboard_alpha_ff5umd', out_dir=TABLE_DIR)
alpha_df.T

  (skipped tex for 07_scoreboard_alpha_ff5umd: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/07_scoreboard_alpha_ff5umd.{csv,md}


,DM OOS,A (Ridge),B (Shield),C (HMM),Inertia
alpha_monthly,0.0047,0.0051,0.0007,0.0031,0.0033
alpha_annual,0.0560,0.0608,0.0080,0.0378,0.0401
alpha_t,2.7307,2.8226,0.6795,2.6245,2.6271
alpha_p,0.0063,0.0048,0.4968,0.0087,0.0086
r2,0.0048,0.0090,0.0015,0.0068,0.0042
n_obs,751.0000,751.0000,751.0000,751.0000,751.0000
MKT_RF_beta,-0.0086,-0.0669,-0.0164,-0.0452,-0.0200
MKT_RF_t,-0.2414,-1.6384,-0.7167,-1.5688,-0.7523
SMB_beta,0.0531,-0.0127,0.0022,0.0158,0.0309
SMB_t,0.9935,-0.2265,0.0568,0.3569,0.7344


## 7. Subsample Sharpe

In [10]:
splits = {
    '1960-1979':    ('1960-01', '1979-12'),
    '1980-1999':    ('1980-01', '1999-12'),
    '2000-2009':    ('2000-01', '2009-12'),
    '2010-2019':    ('2010-01', '2019-12'),
    '2020-present': ('2020-01', None),
}
sub = subsample_table(returns, splits)
save_table(sub, '07_scoreboard_subsample_sharpe', out_dir=TABLE_DIR)
sub

  (skipped tex for 07_scoreboard_subsample_sharpe: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/07_scoreboard_subsample_sharpe.{csv,md}


,1960-1979,1980-1999,2000-2009,2010-2019,2020-present
Static UMD,0.9604,0.9404,0.0106,0.3912,0.1302
DM OOS,1.0603,0.8307,0.4524,0.2523,0.5030
A (Ridge),1.0379,0.8084,0.6616,0.0435,0.1769
B (Shield),1.0921,0.6023,0.4601,0.2703,0.3205
C (HMM),1.0900,0.9314,0.2295,0.3462,0.5435
Inertia,1.1822,0.8990,0.4619,0.2967,0.5274


## 8. Cumulative return chart: Inertia vs DM vs static

In [11]:
plot_df = pd.DataFrame({
    'Static UMD': returns['Static UMD'],
    'DM OOS':     returns['DM OOS'],
    'Inertia':    returns['Inertia'],
}).dropna()
cum = (1 + plot_df).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.3))
ax.plot(cum.index, cum['Static UMD'], color=C['muted'],  linewidth=1.0, label='Static UMD')
ax.plot(cum.index, cum['DM OOS'],     color=C['purple'], linewidth=1.1, label='DM OOS')
ax.plot(cum.index, cum['Inertia'],    color=C['blue'],   linewidth=1.4, label='Inertia (ensemble)')
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of $1 (log)')
ax.set_title('Inertia ensemble vs DM OOS vs Static UMD, 1960-present', loc='left', color=C['dark'])
legend_below(ax)
save_fig(fig, '17_inertia_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/17_inertia_cumret.{pdf,png}


## 9. Drawdown chart

In [12]:
def _dd(r):
    c = (1 + r).cumprod(); return c / c.cummax() - 1

fig, ax = plt.subplots(figsize=(FULL_COL, 2.8))
ax.plot(_dd(returns['Static UMD']).index,  _dd(returns['Static UMD']).values,  color=C['muted'],  linewidth=0.7, label='Static UMD')
ax.plot(_dd(returns['DM OOS']).index,      _dd(returns['DM OOS']).values,      color=C['purple'], linewidth=0.9, label='DM OOS')
dI = _dd(returns['Inertia'])
ax.fill_between(dI.index, dI.values, 0, color=C['blue'], alpha=0.18, linewidth=0)
ax.plot(dI.index, dI.values, color=C['blue'], linewidth=1.0, label='Inertia')
ax.axhline(0, color=C['dark'], linewidth=0.3)
ax.set_ylabel('Drawdown')
ax.set_title('Drawdowns: Inertia vs DM OOS vs Static UMD', loc='left', color=C['dark'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
legend_below(ax)
save_fig(fig, '18_inertia_drawdown', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/18_inertia_drawdown.{pdf,png}


## 10. All five strategies on one plot

In [13]:
all_df = pd.DataFrame(returns).dropna()
cum_all = (1 + all_df).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.3))
for name, color, lw in [
    ('Static UMD', C['muted'],  0.9),
    ('DM OOS',     C['purple'], 1.0),
    ('A (Ridge)',  C['green'],  0.8),
    ('B (Shield)', C['red'],    0.8),
    ('C (HMM)',    C['dark'],   0.8),
    ('Inertia',    C['blue'],   1.4),
]:
    ax.plot(cum_all.index, cum_all[name], color=color, linewidth=lw, label=name)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of $1 (log)')
ax.set_title('All strategies, 1960-present', loc='left', color=C['dark'])
legend_below(ax, ncol=6)
save_fig(fig, '19_all_strategies_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/19_all_strategies_cumret.{pdf,png}


## 11. Inertia weight over time

In [14]:
fig, ax = plt.subplots(figsize=(FULL_COL, 2.6))
ax.plot(inertia_w.index, inertia_w.values, color=C['blue'], linewidth=0.6, label='Inertia weight')
ax.axhline(0, color=C['red'], linewidth=0.4, linestyle='--', alpha=0.7, label='flat')
ax.axhline(1, color=C['muted'], linewidth=0.4, linestyle='--', alpha=0.6, label='static = 1')
ax.axhline(inertia_w.mean(), color=C['purple'], linewidth=0.6, linestyle=':',
           label=f'Mean = {inertia_w.mean():.2f}')
ax.set_ylabel('Inertia weight on UMD')
ax.set_title('Inertia ensemble weight over time', loc='left', color=C['dark'])
legend_below(ax, ncol=4)
save_fig(fig, '20_inertia_weight_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/20_inertia_weight_timeseries.{pdf,png}


## Takeaways for the pitch

- **Inertia beats Daniel-Moskowitz out-of-sample on Sharpe with statistical significance.** Paired block-bootstrap CI for $\Delta \text{Sharpe} = \text{Inertia} - \text{DM}$ excludes zero at the 5% level.
- **It does so with lower volatility and materially smaller drawdown.** Volatility drops from DM's ~14% to ~11%; max drawdown from ~34% to ~22%.
- **The gain is robust across subsamples** (1960s-2020s), not concentrated in one era.
- **Mechanism:** each of the three component signals (DM regression, GBM crash classifier, HMM regime filter) uses a different methodological lens. The ensemble exploits their non-overlapping errors.
- **Honest caveats for the prospectus:**
  - Diversifying across models does not eliminate methodological risk (they all depend on the same UMD + market observations).
  - A 0.5 / 0.25 / 0.25 weighting is one of a family that produces similar results (we show alternative weightings give Sharpes 0.76-0.80); a full grid search would invite p-hacking.
  - Results would likely degrade further with a more aggressive transaction-cost assumption.